# Lab 3 : A simple AWS agent

*Week 5 · Utrains LLMOps 8 Week Course*

Run each cell from the top. Read what it prints before you run the next cell.




## Objective

Labs 1 and 2 used a small **order-desk** example (mock data in Python).

This lab uses the **same agent idea** on a real system: **Amazon Web Services (AWS)**.

You will build three tools:

| Tool | What it answers |
|------|-----------------|
| `count_ec2_instances` | How many EC2 instances are there? |
| `count_s3_buckets` | How many S3 buckets are there? |
| `check_iam_user` | Is this IAM user present? (for example John) |

Then a user asks a **generic** question in plain English. The model **picks the right tool**, your code **runs it against AWS**, and the model **answers from the tool result**.

Examples:

- "How many EC2 instances are present?"
- "How many S3 buckets are present?"
- "Is user John present?"

That is an agentic system: **question → select tool → run tool → answer**.

### What you need

1. Same `week05/.env` with `ANTHROPIC_API_KEY`.
2. Also add your AWS keys (see Step 1).

**Safety.** Prefer an IAM user with **read-only** access (list/describe EC2, list S3 buckets, get IAM user). Never commit `.env`.


### Step 1. Load keys, model, and AWS clients

`python-dotenv` loads secrets from `.env`.

`boto3` is the official AWS SDK for Python. It reads:

- `AWS_ACCESS_KEY_ID`
- `AWS_SECRET_ACCESS_KEY`
- `AWS_DEFAULT_REGION`

Put those in `week05/.env` next to your Anthropic key.


In [6]:
from dotenv import load_dotenv
from langchain_anthropic import ChatAnthropic
import boto3

load_dotenv()

llm = ChatAnthropic(model="claude-haiku-4-5", temperature=0)

# boto3 reads AWS keys from .env (loaded above)
ec2 = boto3.client("ec2")
s3 = boto3.client("s3")
iam = boto3.client("iam")

print("Model ready.")
print("AWS clients ready.")


Model ready.
AWS clients ready.


### Step 2. Three simple AWS tools

Each tool is a normal Python function. The docstring tells the model **when** to use it.




In [8]:
from langchain_core.tools import tool


@tool
def count_ec2_instances() -> dict:
    """Return how many EC2 instances are in this AWS account and region."""
    response = ec2.describe_instances()
    count = 0
    for reservation in response.get("Reservations", []):
        count += len(reservation.get("Instances", []))
    return {"ec2_instance_count": count}


@tool
def count_s3_buckets() -> dict:
    """Return how many S3 buckets are in this AWS account."""
    response = s3.list_buckets()
    buckets = response.get("Buckets", [])
    return {"s3_bucket_count": len(buckets)}


@tool
def check_iam_user(user_name: str) -> dict:
    """Check if an IAM user exists. Example: user_name='John'."""
    try:
        response = iam.get_user(UserName=user_name)
        user = response["User"]
        return {"exists": True, "user_name": user["UserName"], "arn": user["Arn"]}
    except Exception:
        return {"exists": False, "user_name": user_name}


TOOLS = [count_ec2_instances, count_s3_buckets, check_iam_user]



### Step 3. The agent path (same idea as Lab 2)

One cell. Four steps:

1. **SELECT TOOL** — model chooses EC2, S3, or IAM  
2. **RUN TOOL** — your code calls AWS  
3. **SEND RESULT BACK** — tool output goes into the conversation  
4. **FINAL ANSWER** — model answers the user  

Start with a generic EC2 question. Then change only the `question` string to try S3 or IAM.


In [9]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage

llm_with_tools = llm.bind_tools(TOOLS)

# --- Change this question to try EC2 / S3 / IAM ---
question = "How many EC2 instances are present?"

messages = [
    SystemMessage(
        content=(
            "You are an AWS assistant. "
            "Use tools for EC2, S3, and IAM questions. "
            "Answer briefly using only tool results. "
            "Do not invent AWS data."
        )
    ),
    HumanMessage(content=question),
]

# 1) SELECT TOOL
ai = llm_with_tools.invoke(messages)
messages.append(ai)

print("1) SELECT TOOL")
print("   tool_calls:", ai.tool_calls)
print()

if not ai.tool_calls:
    print("Model answered without a tool:")
    print(ai.content)
else:
    # 2) RUN TOOL
    call = ai.tool_calls[0]
    name = call["name"]
    args = call["args"]

    if name == "count_ec2_instances":
        result = count_ec2_instances.invoke(args)
    elif name == "count_s3_buckets":
        result = count_s3_buckets.invoke(args)
    elif name == "check_iam_user":
        result = check_iam_user.invoke(args)
    else:
        result = {"error": f"Unknown tool: {name}"}

    print("2) RUN TOOL")
    print("   name:", name)
    print("   args:", args)
    print("   result:", result)
    print()

    # 3) SEND RESULT BACK
    messages.append(
        ToolMessage(content=str(result), tool_call_id=call["id"])
    )
    print("3) SEND RESULT BACK")
    print()

    # 4) FINAL ANSWER
    final = llm_with_tools.invoke(messages)
    print("4) FINAL ANSWER")
    print("  ", final.content)


1) SELECT TOOL
   tool_calls: [{'name': 'count_ec2_instances', 'args': {}, 'id': 'toolu_01C4Fxur3kAFwKMkwPugvH6Y', 'type': 'tool_call'}]

2) RUN TOOL
   name: count_ec2_instances
   args: {}
   result: {'ec2_instance_count': 2}

3) SEND RESULT BACK

4) FINAL ANSWER
   There are **2 EC2 instances** in this AWS account and region.


### Step 4. Try the other tools

Re-run Step 3 after changing `question` to:

- **S3:** `How many S3 buckets are present?`
- **IAM:** `Is user John present?`

(Use a real IAM user name from your account if John does not exist.)

### What you should remember

1. Lab 1: the model **asks** for a tool.  
2. Lab 2: your code **runs** the tool and the model **answers**.  
3. Lab 3: the same pattern, but tools talk to a **real system** (AWS).

The concept does not change. Only the data source changes.
